# MaintAI — Adaptation légère (LoRA / QLoRA) d'un petit LLM sur MaintIE

**Réglages Kaggle :** Accelerator = **GPU T4 x2**, Internet = **On**, et ajouter le dataset `maintai` (Add Input).

Chaque section est indépendante : vous pouvez ne lancer que celle qui vous est attribuée.

In [ ]:
# Un seul GPU : mesures de mémoire correctes et comportement identique pour tout le groupe
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
!pip install -q 'peft>=0.12' bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copie du code et des données dans le dossier de travail
import glob, shutil
src = os.path.dirname(glob.glob('/kaggle/input/**/infer.py', recursive=True)[0])
shutil.copytree(src, '/kaggle/working/maintai', dirs_exist_ok=True)
%cd /kaggle/working/maintai
!ls data && wc -l data/*.jsonl

## 0. Vérification rapide (≈ 5 min) — à lancer en premier

In [ ]:
!python infer.py --model Qwen/Qwen2.5-0.5B-Instruct --shots 5 --limit 10 --out results/check.jsonl

## 1. Modèle 0.5B : baseline puis LoRA

In [ ]:
M = 'Qwen/Qwen2.5-0.5B-Instruct'
!python infer.py --model {M} --shots 0 --out results/base05_zs.jsonl
!python infer.py --model {M} --shots 5 --out results/base05_fs.jsonl

In [ ]:
!python train_lora.py --model {M} --out runs/lora05
!python infer.py --model {M} --adapter runs/lora05 --shots 0 --out results/lora05.jsonl

## 2. Modèle 3B : baseline, LoRA, puis QLoRA (optionnel)

In [ ]:
M3 = 'Qwen/Qwen2.5-3B-Instruct'
!python infer.py --model {M3} --shots 0 --out results/base3b_zs.jsonl
!python infer.py --model {M3} --shots 5 --out results/base3b_fs.jsonl

In [ ]:
!python train_lora.py --model {M3} --batch 4 --out runs/lora3b
!python infer.py --model {M3} --adapter runs/lora3b --shots 0 --out results/lora3b.jsonl

In [ ]:
# QLoRA : base quantifiée en 4 bits à l'entraînement ET à l'inférence
!python train_lora.py --model {M3} --qlora --batch 4 --out runs/qlora3b
!python infer.py --model {M3} --adapter runs/qlora3b --load-4bit --shots 0 --out results/qlora3b.jsonl

## 3. Évaluation et tableau comparatif

In [ ]:
import glob
for f in sorted(glob.glob('results/*.jsonl')):
    name = os.path.basename(f)[:-6]
    if name == 'check':
        continue
    !python evaluate.py --gold data/test.jsonl --pred {f} --out results/{name}_metrics.json > /dev/null
!python summarize.py --out results/tableau.csv

In [ ]:
import pandas as pd
pd.read_csv('results/tableau.csv')

## 4. Sauvegarde
Les fichiers de `/kaggle/working` sont conservés dans l'onglet **Output** quand vous faites **Save Version → Save & Run All**. Pour garder les adaptateurs entre sessions, publiez l'output comme dataset ou téléchargez `runs/` et `results/`.

In [ ]:
!zip -qr /kaggle/working/maintai_outputs.zip results runs -x 'runs/*/checkpoint*'
!ls -lh /kaggle/working/maintai_outputs.zip